In [2]:
import sys
import os
# Trỏ đường dẫn gốc về thư mục project để import file từ src (nếu cần)
sys.path.append(os.path.abspath('..')) 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import glob


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
data_dir = '/content/drive/MyDrive/Colab-Notebooks/processed_data'
files = sorted(glob.glob(f'{data_dir}/House2_part1.csv'))

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(f"Tổng số dòng: {len(df)}")
print(df.head())

Tổng số dòng: 1000000
                  Time        Unix  Aggregate  Appliance1  Appliance2  \
0  2013-09-17 22:08:11  1379455691        695          88           0   
1  2013-09-17 22:08:18  1379455698        694          88           0   
2  2013-09-17 22:08:26  1379455706        694          88           0   
3  2013-09-17 22:08:34  1379455714        702          88           0   
4  2013-09-17 22:08:42  1379455722        700          88           0   

   Appliance3  Appliance4  Appliance5  Appliance6  Appliance7  Appliance8  \
0           0           0           0           0           0           0   
1           0           0           0           0           0           0   
2           0           0           0           0           0           0   
3           0           0           0           0           0           0   
4           0           0           0           0           0           0   

   Appliance9  
0           0  
1           0  
2           0  
3           

In [17]:
# 1. Cuối tuần (Thứ 7, CN) -> Trả về 1 (Có) hoặc 0 (Không)
df['Is_Weekend'] = df['Time'].dt.dayofweek.isin([5, 6]).astype(int)

# 2. Chia 4 buổi trong ngày
def get_time_of_day(hour):
    if 5 <= hour < 8: return 1   # Sáng (Morning)
    elif 8 <= hour < 17: return 2  # Ngày (Daytime)
    elif 17 <= hour < 22: return 3 # Tối (Evening)
    else: return 0                 # Đêm (Night)

df['Time_of_Day'] = df['Hour'].apply(get_time_of_day)

# 2. Định nghĩa tập X (Time features + Aggregate hiện tại)
X = df[['Hour', 'DayOfWeek', 'Is_Weekend', 'Time_of_Day', 'Aggregate']]
time_index = df['Time'] # Giữ lại để lát vẽ biểu đồ

# 3. Định nghĩa tập y (10 cột)
list_9_appliances = ['Appliance1', 'Appliance2', 'Appliance3', 'Appliance4', 'Appliance5', 'Appliance6', 'Appliance7', 'Appliance8', 'Appliance9']

# Tính cột thứ 10: Unknown_Appliance
# Lưu ý: Đôi khi do nhiễu đo lường, tổng 9 thiết bị có thể lớn hơn Aggregate sinh ra số âm. 
# Dùng .clip(lower=0) để đưa các số âm về 0 cho chuẩn thực tế.
df['Unknown_Appliance'] = (df['Aggregate'] - df[list_9_appliances].sum(axis=1)).clip(lower=0)

y = df[list_9_appliances + ['Unknown_Appliance']]

# 4. Chia Train/Test (Vì là chuỗi thời gian, ta KHÔNG xáo trộn dữ liệu - shuffle=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
time_train, time_test = train_test_split(time_index, test_size=0.2, shuffle=False)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (800000, 5), y_train shape: (800000, 10)


In [18]:
# Sử dụng Random Forest, có thể tuning n_estimators hoặc max_depth
model = RandomForestRegressor(n_estimators=100, max_depth=10,min_samples_leaf=5,max_features=None, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Dự đoán
y_pred = model.predict(X_test)

# Bất kỳ dự đoán nào < 5W (hoặc một ngưỡng nhỏ), ta ép thẳng về 0W
THRESHOLD = 5.0
y_pred[y_pred < THRESHOLD] = 0.0






In [19]:
def calculate_nilm_metrics(y_true, y_pred, appliance_names):
    """
    Tính toán các chỉ số Energy-based Precision, Recall, F1-score và NEP
    dựa trên công thức học thuật.
    """
    # Đảm bảo đầu vào là numpy array để tránh lỗi lệch index của Pandas
    y_t = np.array(y_true)
    y_p = np.array(y_pred)
    
    # Cộng thêm 1e-9 (số cực nhỏ) vào mẫu số để tránh lỗi Division by Zero (chia cho 0)
    # vì có những thiết bị tắt hoàn toàn trong tập test (tổng năng lượng = 0)
    eps = 1e-9 
    
    # Tính tử số chung của Precision và Recall: sum(min(y_pred, y_true))
    min_power = np.minimum(y_p, y_t)
    sum_min_power = np.sum(min_power, axis=0) # Cộng dồn theo trục thời gian (T)
    
    # Tính mẫu số
    sum_pred = np.sum(y_p, axis=0) # Tổng điện dự đoán
    sum_true = np.sum(y_t, axis=0) # Tổng điện thực tế
    
    # 1. Energy-based Precision (PE)
    P_E = sum_min_power / (sum_pred + eps)
    
    # 2. Energy-based Recall (RE)
    R_E = sum_min_power / (sum_true + eps)
    
    # 3. Energy-based F1-score (FE)
    F1_E = 2 * (P_E * R_E) / (P_E + R_E + eps)
    
    # 4. Normalized Error in Assigned Power (NEP)
    sum_abs_error = np.sum(np.abs(y_t - y_p), axis=0)
    NEP = sum_abs_error / (sum_true + eps)
    
    # --- Đóng gói kết quả thành Bảng (DataFrame) cho dễ nhìn ---
    results_df = pd.DataFrame({
        'Appliance': appliance_names,
        'Precision (PE)': np.round(P_E, 4),
        'Recall (RE)': np.round(R_E, 4),
        'F1-Score (FE)': np.round(F1_E, 4),
        'NEP': np.round(NEP, 4)
    })
    
    # Thêm 1 dòng tính điểm Trung bình (Macro Average) của cả nhà
    mean_row = pd.DataFrame({
        'Appliance': ['--- AVERAGE ---'],
        'Precision (PE)': [np.round(np.mean(P_E), 4)],
        'Recall (RE)': [np.round(np.mean(R_E), 4)],
        'F1-Score (FE)': [np.round(np.mean(F1_E), 4)],
        'NEP': [np.round(np.mean(NEP), 4)]
    })
    
    results_df = pd.concat([results_df, mean_row], ignore_index=True)
    return results_df

# Danh sách tên 10 thiết bị (lấy từ các bước trước)
appliance_names = list_9_appliances + ['Unknown_Appliance']

# Gọi hàm tính điểm
metrics_table = calculate_nilm_metrics(y_test, y_pred, appliance_names)

# Hiển thị bảng điểm
print("BẢNG ĐÁNH GIÁ MÔ HÌNH DỰA TRÊN NĂNG LƯỢNG (ENERGY-BASED METRICS):")
display(metrics_table)


BẢNG ĐÁNH GIÁ MÔ HÌNH DỰA TRÊN NĂNG LƯỢNG (ENERGY-BASED METRICS):


,Appliance,Precision (PE),Recall (RE),F1-Score (FE),NEP
0,Appliance1,0.6361,0.6902,0.6621,0.7046
1,Appliance2,0.2257,0.2686,0.2453,1.6530
2,Appliance3,0.5506,0.5569,0.5537,0.8976
3,Appliance4,0.2525,0.3778,0.3027,1.7408
4,Appliance5,0.3958,0.2773,0.3261,1.1460
5,Appliance6,0.5233,0.5141,0.5186,0.9543
6,Appliance7,0.0850,0.0188,0.0308,1.1838
7,Appliance8,0.7296,0.6817,0.7049,0.5709
8,Appliance9,0.0151,0.0486,0.0230,4.1214
9,Unknown_Appliance,0.8873,0.8664,0.8767,0.2436


In [ ]:
def plot_10_appliances(y_test, y_pred, time_test):
    appliance_names = list_9_appliances + ['Unknown_Appliance']
    
    # Tạo 10 subplots (5 hàng, 2 cột)
    fig, axes = plt.subplots(5, 2, figsize=(16, 20))
    axes = axes.flatten()
    
    # Lấy khoảng 1000 điểm dữ liệu đầu tiên của tập test để biểu đồ không bị rối mớ bong bong
    plot_length = 1000 
    
    for i in range(10):
        # Vẽ giá trị thực tế
        axes[i].plot(time_test.iloc[:plot_length], y_test.iloc[:plot_length, i], 
                     label='Thực tế (Actual)', color='blue', alpha=0.6)
        # Vẽ giá trị dự đoán
        axes[i].plot(time_test.iloc[:plot_length], y_pred[:plot_length, i], 
                     label='Dự đoán (Predicted)', color='red', alpha=0.6, linestyle='--')
        
        axes[i].set_title(f'Thiết bị: {appliance_names[i]}')
        axes[i].set_ylabel('Công suất (W)')
        axes[i].legend(loc='upper right')
        
        # Format lại trục X cho dễ nhìn
        plt.setp(axes[i].xaxis.get_majorticklabels(), rotation=45)

    plt.tight_layout()
    plt.show()

plot_10_appliances(y_test, y_pred, time_test)